In [1]:
# Add this at the start to mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Then change results_dir to save to Drive:
# results_dir = "/content/drive/My Drive/thoth/research/papers/globecom26/FedAKD"

results_dir = "results"

# Imports and setup

In [12]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout, MaxPooling1D, Conv1D, Flatten
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping



import os
from os.path import join
import warnings
import tensorflow as tf 
from tensorflow import keras
from tensorflow.keras import layers

from datasets import load_dataset
from scipy.signal import resample

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [3]:
# GPU Configuration

# Enable GPU memory growth for efficient training
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    for device in physical_devices:
        tf.config.experimental.set_memory_growth(device, True)
print(f"GPUs available: {len(physical_devices)}")

GPUs available: 0


E0000 00:00:1777318966.353272   11628 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1777318966.376537   11628 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


# Functions and classes

In [5]:



# ============ CSI PREPROCESSING FUNCTIONS ============

def parse_csi_data(data_str):
    """Parse CSI data string to complex I/Q array and extract 52 LLTF subcarriers"""
    # Handle non-string values (NaN, float, etc.)
    if not isinstance(data_str, str):
        return None
    
    try:
        # Clean the string and parse
        cleaned = data_str.strip().strip('[]"')
        if not cleaned:
            return None
        values = list(map(float, cleaned.split(',')))
        
        # Need at least 128 values for 64 complex subcarriers
        if len(values) < 128:
            return None
            
        # Extract I/Q pairs for 64 subcarriers
        complex_csi = np.array([values[i] + 1j*values[i+1] for i in range(0, min(128, len(values)), 2)])
        # Select 52 LLTF subcarriers (remove guard/null subcarriers)
        selected = np.concatenate([complex_csi[6:32], complex_csi[33:59]])  # 52 subcarriers
        return selected
    except (ValueError, IndexError) as e:
        return None

def preprocess_csi_dataframe(df, window_size=100, target_rate=150, variance_window=200):
    """Full preprocessing pipeline for a CSI dataframe"""
    # Check if 'data' column exists
    if 'data' not in df.columns:
        print(f"    Warning: 'data' column not found. Available columns: {list(df.columns)}")
        return np.array([]).reshape(0, window_size, 52)
    
    # Parse CSI data column, filtering out invalid entries
    parsed_data = []
    for row in df['data']:
        result = parse_csi_data(row)
        if result is not None:
            parsed_data.append(result)
    
    if len(parsed_data) == 0:
        return np.array([]).reshape(0, window_size, 52)
    
    csi_data = np.array(parsed_data)
    amplitudes = np.abs(csi_data)  # Use amplitude
    
    # Resample to uniform rate (original is ~100-200 Hz irregular)
    n_samples = int(len(amplitudes) * target_rate / 100)
    if n_samples < window_size:
        n_samples = len(amplitudes)
    resampled = resample(amplitudes, n_samples, axis=0)
    
    # Rolling variance feature extraction
    features = []
    for i in range(len(resampled)):
        start_idx = max(0, i - variance_window + 1)
        features.append(np.var(resampled[start_idx:i+1], axis=0))
    features = np.array(features)
    
    # Windowing: segment into fixed-length windows
    n_windows = len(features) // window_size
    if n_windows == 0:
        return np.array([]).reshape(0, window_size, 52)
    windows = features[:n_windows * window_size].reshape(n_windows, window_size, -1)
    return windows

def load_and_preprocess_homeoccupancy(window_size=100):
    """Load HomeOccupancy dataset from HuggingFace and preprocess"""
    from huggingface_hub import hf_hub_download
    
    print("Loading HomeOccupancy dataset from HuggingFace...")
    
    occupancy_labels = {'empty': 0, 'sleep': 1, 'work': 2}
    
    x_train_list, y_train_list = [], []
    x_test_list, y_test_list = [], []
    
    for label_name, label_id in occupancy_labels.items():
        # Training sessions (1 and 2)
        for session in [1, 2]:
            file_name = f'{label_name}_{session}.csv'
            try:
                file_path = hf_hub_download(repo_id="gadgadgad/HomeOccupancy", 
                                           filename=file_name, repo_type="dataset")
                df = pd.read_csv(file_path, on_bad_lines='skip')
                windows = preprocess_csi_dataframe(df, window_size=window_size)
                if len(windows) > 0:
                    x_train_list.append(windows)
                    y_train_list.append(np.full(len(windows), label_id))
                    print(f"  {file_name}: {len(windows)} windows")
                else:
                    print(f"  {file_name}: 0 windows (no valid CSI data)")
            except Exception as e:
                print(f"  Warning: Could not load {file_name}: {e}")
        
        # Test session (3)
        file_name = f'{label_name}_3.csv'
        try:
            file_path = hf_hub_download(repo_id="gadgadgad/HomeOccupancy", 
                                       filename=file_name, repo_type="dataset")
            df = pd.read_csv(file_path, on_bad_lines='skip')
            windows = preprocess_csi_dataframe(df, window_size=window_size)
            if len(windows) > 0:
                x_test_list.append(windows)
                y_test_list.append(np.full(len(windows), label_id))
                print(f"  {file_name} (test): {len(windows)} windows")
            else:
                print(f"  {file_name} (test): 0 windows (no valid CSI data)")
        except Exception as e:
            print(f"  Warning: Could not load {file_name}: {e}")
    
    x_train = np.concatenate(x_train_list) if x_train_list else np.array([]).reshape(0, window_size, 52)
    y_train = np.concatenate(y_train_list) if y_train_list else np.array([])
    x_test = np.concatenate(x_test_list) if x_test_list else np.array([]).reshape(0, window_size, 52)
    y_test = np.concatenate(y_test_list) if y_test_list else np.array([])
    
    print(f"\nHomeOccupancy loaded: {len(x_train)} train windows, {len(x_test)} test windows")
    return x_train, y_train, x_test, y_test, list(occupancy_labels.keys())

def load_and_preprocess_homehar(window_size=100):
    """Load HomeHAR dataset from HuggingFace and preprocess for public distillation data"""
    from huggingface_hub import hf_hub_download
    
    print("Loading HomeHAR dataset from HuggingFace...")
    
    har_labels = {'drink': 0, 'eat': 1, 'empty': 2, 'sleep': 3, 'smoke': 4, 'watch': 5, 'work': 6}
    
    pub_x_list, pub_y_list = [], []
    
    # Use data1 and data2 sessions for public data
    for session in ['data1', 'data2']:
        for label_name, label_id in har_labels.items():
            # Try different filename formats
            possible_files = [
                f'{session}/{label_name}.csv',
                f'{session}_{label_name}.csv',
                f'{label_name}_{session}.csv'
            ]
            
            for file_name in possible_files:
                try:
                    file_path = hf_hub_download(repo_id="gadgadgad/HomeHAR", 
                                               filename=file_name, repo_type="dataset")
                    df = pd.read_csv(file_path, on_bad_lines='skip')
                    windows = preprocess_csi_dataframe(df, window_size=window_size)
                    if len(windows) > 0:
                        pub_x_list.append(windows)
                        pub_y_list.append(np.full(len(windows), label_id))
                        print(f"  {file_name}: {len(windows)} windows")
                    break
                except Exception:
                    continue
    
    pub_x = np.concatenate(pub_x_list) if pub_x_list else np.array([]).reshape(0, window_size, 52)
    pub_y = np.concatenate(pub_y_list) if pub_y_list else np.array([])
    
    print(f"\nHomeHAR loaded: {len(pub_x)} public windows")
    return pub_x, pub_y, list(har_labels.keys())


# ============ ORIGINAL FUNCTIONS (kept for compatibility) ============

def get_data (data_dir, label, as_np = False, as_df = False) : 

    experiments = [] 
    data = [] 
    with open(data_dir, 'r') as f : 
        for line in f.readlines() : 
            if line.startswith('month') : 
                if len(data) : 
                    experiments.append(data) 
                    data = [] 
            data.append(line) 
        if len(data) : 
            experiments.append(data) 
    
    if as_np or as_df : 
        np_exps = [] 
        for i, exp in enumerate(experiments) : 
            _, data = put_experiment_data_to_np(exp, label = label)  
            np_exps.append(data) 
        np_exp = np.concatenate(np_exps) 

        if as_df : 
            dataframe_dict = {
                'hr': np_exp[:, 0],
                'gryo_x': np_exp[:, 1], 
                'gyro_y': np_exp[:, 2], 
                'gyro_z': np_exp[:, 3],
                'timestamp': np_exp[:, 4], 
                'label': np_exp[:, 5]}
            df = pd.DataFrame(dataframe_dict)
            return df 
         
        return np_exp 


    return experiments 



def put_experiment_data_to_np(exp, label = None) : 

    def get_first_hr(exp) : 
        for i in range(1, len(exp)):
            line = exp[i]
            vars = line.split(',') 
            if len(vars) == 2 : 
                return int(vars[0])
    hr = get_first_hr(exp) 
    np_data = []
    for i in range(1, len(exp)):
        line = exp[i]
        vars = line.split(',') 
        if len(vars) == 2 : 
            if (int(vars[0])  < 0) : continue 
            hr = (int(vars[0]) + hr) // 2
        elif len(vars) == 4 : 
            gryo_vars = list(map(int, vars[:3])) 
            if label is not None : 
                data = np.array([hr, *gryo_vars, int(vars[3].split('.')[0]), label])
            else : 
                data = np.array([hr, *gryo_vars, int(vars[3].split('.')[0])])
            np_data.append(data)      
    return exp[0], np.array(np_data) 



def get_models(p, configurations) : 

    learning_rate = p['learning_rate']
    seq_len = p['seq_len']
    n_features = p['n_features']
    n_classes = p['n_classes']

    filters1, filters2, filters3 = configurations['filters']
    lstm_units1, lstm_units2 = configurations['lstm_units']
    ks1, ks2, ks3 = configurations['kernel_size']

    K.clear_session() 
    #__________________________Inputs
    inp = Input(shape = (seq_len, n_features))
    #__________________________layers
    conv1 = Conv1D(filters = filters1 , kernel_size = 20, dilation_rate = 2, activation = 'relu')(inp)
    conv2 = Conv1D(filters = filters2 , kernel_size = 20, dilation_rate = 2, activation = 'relu')(conv1)
    conv3 = Conv1D(filters = filters3 , kernel_size = 10, dilation_rate = 2, activation = 'relu')(conv2)
    lstm1 = LSTM(lstm_units1, return_sequences = True, activation = 'tanh')(conv3)
    lstm2 = LSTM(lstm_units2, return_sequences = False, activation = 'tanh')(lstm1) 
    lstm_d = Dropout(0.1)(lstm2)
    dense1 = Dense(10)(lstm_d)
    dense2 = Dense(n_classes)(dense1)
    dense2 = Activation('softmax')(dense2)
    model_A = models.Model(inputs = inp, outputs = dense2)

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    loss = tf.keras.losses.CategoricalCrossentropy(from_logits=False)
    model_A.compile(optimizer= optimizer, loss = loss, metrics = ["accuracy"])
    
    # remove the last layer and compile with a different loss function
    model_B = models.Model(inputs = model_A.inputs, outputs = model_A.layers[-2].output)
    model_B.compile(optimizer = optimizer, loss = "mean_absolute_error")

    return model_A, model_B


def split_dataset(x, y, samples_per_class, n_models, include_classes, to_categorical_flag = False, n_classes_total = None) : 
    datasets = [None]*n_models 
    labels = [None]*n_models 

    sample_indecies = [None]*n_models 
    n_classes = n_classes_total if n_classes_total is not None else len(np.unique(y))
    combined_idx = np.array([], dtype = np.int32) 
     
    all_classes = list(np.unique(y))

    for label in all_classes :
        idx = np.where(y == label)[0]
        n_available = len(idx)
        n_needed = samples_per_class * n_models
        idx = np.random.choice(idx, min(n_needed, n_available * n_models), replace = True) 
        # Pad if needed
        while len(idx) < n_needed:
            idx = np.concatenate([idx, np.random.choice(np.where(y == label)[0], min(samples_per_class, n_available), replace=True)])
        idx = idx[:n_needed]
        
        combined_idx = np.r_[combined_idx, idx]
        for i in range(n_models) :
            if include_classes != 'all' : 
                if label not in include_classes[i] :
                    continue
            if datasets[i] is None :
                datasets[i] = [x[idx[i*samples_per_class : (i+1)*samples_per_class]]]
                labels[i] = [y[idx[i*samples_per_class : (i+1)*samples_per_class]]]
            else : 
                datasets[i].append( x[idx[i*samples_per_class : (i+1)*samples_per_class]])
                labels[i].append(y[idx[i*samples_per_class : (i+1)*samples_per_class]])
  
    for i in range(n_models) : 
        if datasets[i] is not None:
            datasets[i] = np.concatenate(datasets[i])
            labels[i] = np.concatenate(labels[i])
        else:
            datasets[i] = np.array([])
            labels[i] = np.array([])
    

    total_datasets = x[combined_idx]
    total_labels = y[combined_idx]

    
    if to_categorical_flag : 
        for i, l in enumerate(labels): 
            if len(l) > 0:
                labels[i] = tf.keras.utils.to_categorical(l, num_classes = n_classes)       
        total_labels = tf.keras.utils.to_categorical(total_labels, num_classes = n_classes)
    
    return datasets, labels, total_datasets, total_labels 



def get_optimizer(hp) : 
    optimizer_name = hp['optimizer'].lower()
    if optimizer_name == 'adam' :
        return tf.keras.optimizers.Adam(learning_rate=hp['learning_rate'])
    elif optimizer_name == 'sgd' :
        return tf.keras.optimizers.SGD(learning_rate=hp['learning_rate'])
    elif optimizer_name == 'rmsprop' :
        return tf.keras.optimizers.RMSprop(learning_rate=hp['learning_rate'])
    else : 
        raise Exception('Optimizer not supported')


def get_model(n_classes, input_shape, model_configurations) : 
    """
    Create a model for CSI data with Conv1D architecture.
    input_shape should be (window_size, n_features) e.g., (100, 52)
    """
    ub, ua, dropout_rate = model_configurations
    act1, act2 = np.random.choice(["relu", "elu", "selu", "tanh"], 2)
    lr = np.random.choice([1e-3, 1e-4, 1e-5])
    opt = np.random.choice(["adam", "sgd", "rmsprop"])
    optimizer = get_optimizer({"optimizer" : opt, "learning_rate" : lr})
    
    inp = tf.keras.layers.Input(input_shape)
    
    # Check if input is 2D (sequence data) or 1D (flat features)
    if len(input_shape) == 2:
        # Conv1D architecture for sequence data (window_size, n_features)
        y = Conv1D(64, 5, activation='relu', padding='same')(inp)
        y = MaxPooling1D(2)(y)
        y = Conv1D(128, 3, activation='relu', padding='same')(y)
        y = MaxPooling1D(2)(y)
        y = Flatten()(y)
        y = Dense(units = ua, activation = act1)(y) 
        y = Dropout(dropout_rate)(y)
        y = Dense(units = ub, activation = act2)(y)
    else:
        # Dense architecture for flat features
        y = Dense(units = ua, activation = act1)(inp) 
        y = Dropout(dropout_rate)(y)
        y = Dense(units = ub, activation = act2)(y) 
    
    y = Dense(units = n_classes)(y)
    y = tf.keras.layers.Activation('softmax')(y) 

    model_A = tf.keras.models.Model(inputs = inp, outputs = y)

    model_A.compile(optimizer=optimizer,  
                        loss = "categorical_crossentropy",
                        metrics = ["accuracy"])
    model_B = remove_last_layer(model_A) 
    configurations = {
        "act1" : act1,
        "act2" : act2,
        "ua" : ua,
        "ub" : ub,
        "dropout_rate" : dropout_rate,
        "lr" : lr,
        "opt" : opt
        
    }
    return model_A, model_B, configurations 


def remove_last_layer(model, loss = "mean_absolute_error"):
    """
    Input: Keras model, a classification model whose last layer is a softmax activation
    Output: Keras model, the same model with the last softmax activation layer removed,
        while keeping the same parameters 
    """
    
    new_model = tf.keras.models.Model(inputs = model.inputs, outputs = model.layers[-2].output)
    new_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate = 1e-3), 
                      loss = loss)
    
    return new_model


def get_label(filename) : 
    filename = filename.split(".")[0]
    if filename.startswith('study') :
        return 0
    elif filename.startswith('walk') : 
        return 1
    elif filename.startswith('sleep') : 
        return 2
    elif filename.startswith('idle') : 
        return 3
    else :
        return -1

In [6]:



def number_of_parameters(model) : 
    return model.count_params()



def get_tunable_models(hp) : 
    K.clear_session()
    n_conv_layers = hp['n_conv_layers'] 
    n_lstm_layers = hp['n_lstm_layers']
    activation_function = hp['activation_function']
    dropout_rate = hp['dropout_rate']
    conv_filter = hp['conv_filter']
    conv_kernel_size = hp['conv_kernel_size']
    lstm_units = hp['lstm_units']
    optimizer = get_optimizer(hp)
    learning_rate = hp['learning_rate']
    input_shape = hp['input_shape']

    x = Input(shape = input_shape) 
    y = Reshape(input_shape)(x) 
    for i in range(n_conv_layers) : 
        y = Conv1D(conv_filter, conv_kernel_size, activation = activation_function)(y)
        y = MaxPooling1D(2)(y)
    y = Dropout(dropout_rate)(y)
    for i in range(n_lstm_layers) :
        y = LSTM(lstm_units, return_sequences = True)(y)
    lstm_layer = LSTM(lstm_units, return_sequences = False)(y)
    lstm_dropout_layer = Dropout(dropout_rate)(lstm_layer)
    
    dense = Dense(hp['n_classes'])(lstm_dropout_layer)
    output = Activation('softmax')(dense)
    model1 = tf.keras.models.Model(inputs = x, outputs = output)
    model1.compile(optimizer = optimizer, loss= 'categorical_crossentropy', metrics=['accuracy'])
    model2 = tf.keras.models.Model(inputs = x, outputs = dense)
    model2.compile(optimizer = optimizer, loss= 'mean_absolute_error')

    return model1, model2




class Node : 

    model = None 
    carrier_dataset_metadata = None
    carrier_datset = None

    local_target_dataset = None 
    target_validation_set = None 

    target_validation_acc = []
    target_validation_loss = [] 


    def __init__(self, model, local_target_dataset, shared_public_dataset, target_validation_gen) : 
        self.model = model 
        self.local_target_dataset = local_target_dataset
        self.shared_public_dataset = shared_public_dataset
        self.target_validation_gen = target_validation_gen


    def get_training_metadata(self, seed, alpha) : 

        self.seed = seed 
        self.alpha = alpha
        carrier_preds = self.get_carrier_scores() 
        target_performance = self.evaluate_on_validation_set(save = False)[1]
        data_shape = carrier_preds.shape

        # carrier_preds = from_categorical(carrier_preds) 

        # print("carrier_preds:{}  new_labels:{}".format(carrier_preds.shape, new_labels.shape))
        return carrier_preds, target_performance


    def get_carrier_scores(self) : 
        return self.model[1].predict(self.shared_public_dataset[0], batch_size = 32)


    def receive_training_metadata(self, metadata) : 

        self.update_public_dataset_labels(metadata) 


    def update_public_dataset_labels(self, metadata) : 
        new_shared_public_dataset = (self.shared_public_dataset[0], metadata )
        
        self.shared_public_dataset = new_shared_public_dataset


    def evaluate_on_validation_set(self, save = True) : 
        eval_x, eval_y = self.target_validation_gen
        loss, acc = self.model[0].evaluate(eval_x, eval_y, batch_size = 32, verbose = False)
        if save : 
            self.target_validation_loss.append(loss)
            self.target_validation_acc.append(acc)
        return loss, acc

    def train_on_target(self, epochs = 1, use_callbacks = False, verbose = True, logger_file = None, evaluate = False) :
        if use_callbacks : cbs = [EarlyStopping(monitor = 'val_loss', patience = 7, restore_best_weights= True)]
        else : cbs = []
        if logger_file : 
            cbs.append([CSVLogger(filename = logger_file, append = True)])
        
        if evaluate : 
            history = self.model[0].fit(self.local_target_dataset[0], self.local_target_dataset[1], \
                validation_data = self.target_validation_gen, epochs = epochs, callbacks = cbs, verbose = verbose)
        else : 

            history = self.model[0].fit(self.local_target_dataset[0], self.local_target_dataset[1], epochs = epochs, 
                                        callbacks = cbs, verbose = verbose)



    def train_on_public(self, epochs = 1, use_callbacks = False, verbose = True, logger_file = None) : 

        if use_callbacks : cbs = [EarlyStopping(monitor = 'val_loss', patience = 7, restore_best_weights= True)]
        else : cbs = []
        if logger_file : 
            cbs.append([CSVLogger(filename = logger_file, append = True)])
        history = self.model[1].fit(self.shared_public_dataset[0], self.shared_public_dataset[1],
         epochs = epochs, callbacks = cbs, verbose = verbose)



    def save_model(self, model_path) : 
        self.model[0].save(model_path + '_classifier.h5') 
        self.model[1].save(model_path + '_regressor.h5') 


def aggregate_training_metadatas(carrier_labels, target_performance, weighted_averaging = False) : 
    
    
    if weighted_averaging : 
        aggregate_training_metadata = np.average(carrier_labels, weights = target_performance, axis = 0)
    else : 
        aggregate_training_metadata = np.average(carrier_labels, axis = 0) 

    return aggregate_training_metadata


def collect_metadatas(nodes, seed, alpha) : 
    # Collect training metadata
    pub_scores, target_performances = [], []
    for i, node in enumerate(nodes) : 
        training_metadata, target_performance = node.get_training_metadata(seed, alpha) 
        pub_scores.append(training_metadata)
        target_performances.append(target_performance) 
    return pub_scores, target_performances 


class FedAMDNode : 

    model = None 
    carrier_dataset_metadata = None
    carrier_datset = None

    local_target_dataset = None 
    target_validation_set = None 

    target_validation_acc = []
    target_validation_loss = [] 


    def __init__(self, model, local_target_dataset, shared_public_dataset, target_validation_gen) : 
        self.model = model 
        self.local_target_dataset = local_target_dataset
        self.shared_public_dataset = shared_public_dataset
        self.target_validation_gen = target_validation_gen


    def get_training_metadata(self, seed, alpha) : 

        self.seed = seed 
        self.alpha = alpha
        carrier_preds = self.get_carrier_scores() 
        target_performance = self.evaluate_on_validation_set(save = False)[1]
        data_shape = carrier_preds.shape

        # carrier_preds = from_categorical(carrier_preds) 

        # print("carrier_preds:{}  new_labels:{}".format(carrier_preds.shape, new_labels.shape))
        return carrier_preds, target_performance


    def get_carrier_scores(self) : 
        
        x = self.shared_public_dataset[0]
        np.random.seed(self.seed) 
        index = np.random.permutation(len(x))  
        mixed_x = self.alpha * x + (1 - self.alpha) * x[index, ...]
    
        return self.model[1].predict(mixed_x, batch_size = 32)


    def receive_training_metadata(self, metadata) : 

        self.update_public_dataset_labels(metadata) 


    def update_public_dataset_labels(self, metadata) : 
        new_shared_public_dataset = (self.shared_public_dataset[0], metadata )
        
        self.shared_public_dataset = new_shared_public_dataset


    def evaluate_on_validation_set(self, save = True) : 
        eval_x, eval_y = self.target_validation_gen
        loss, acc = self.model[0].evaluate(eval_x, eval_y, batch_size = 32, verbose = False)
        if save : 
            self.target_validation_loss.append(loss)
            self.target_validation_acc.append(acc)
        return loss, acc

    def train_on_target(self, epochs = 1, use_callbacks = False, verbose = True, logger_file = None, evaluate = False) :
        if use_callbacks : cbs = [EarlyStopping(monitor = 'val_loss', patience = 7, restore_best_weights= True)]
        else : cbs = []
        if logger_file : 
            cbs.append([CSVLogger(filename = logger_file, append = True)])
        if evaluate : 
            history = self.model[0].fit(self.local_target_dataset[0], self.local_target_dataset[1], \
                validation_data = self.target_validation_gen, epochs = epochs, callbacks = cbs, verbose = verbose)
        else : 

            history = self.model[0].fit(self.local_target_dataset[0], self.local_target_dataset[1], epochs = epochs, 
                                        callbacks = cbs, verbose = verbose)


    def train_on_public(self, epochs = 1, use_callbacks = False, verbose = True, logger_file = None) : 

        if use_callbacks : cbs = [EarlyStopping(monitor = 'val_loss', patience = 7, restore_best_weights= True)]
        else : cbs = []
        if logger_file : 
            cbs.append([CSVLogger(filename = logger_file, append = True)])
    
        x = self.shared_public_dataset[0]
        np.random.seed(self.seed) 
        index = np.random.permutation(len(x))  
        mixed_x = self.alpha * x + (1 - self.alpha) * x[index, ...]

        history = self.model[1].fit(mixed_x, self.shared_public_dataset[1],
         epochs = epochs, callbacks = cbs, verbose = verbose)



    def save_model(self, model_path) : 
        self.model[0].save(model_path + '_classifier.h5') 
        self.model[1].save(model_path + '_regressor.h5') 




# Data and models setup

In [7]:
experiment_dir = os.path.join(results_dir, 'exp_HomeOccupancy_FedWAKD')
subdirs = ['local_train_iid', 'local_train_noniid', 'central_train_iid', 'central_train_noniid', 'iid', 'noniid']
if not os.path.exists(experiment_dir):
    os.makedirs(experiment_dir)
    for subdir in subdirs : 
        os.makedirs(os.path.join(experiment_dir, subdir))


# ============ PARAMETERS FOR NEW DATASETS ============
n_parties = 20  # 20 clients as requested
n_samples_per_class = 200  # Adjusted for CSI data (fewer samples per window)
window_size = 100  # CSI window size
n_features = 52  # 52 LLTF subcarriers
input_shape = (window_size, n_features)  # (100, 52) for Conv1D
n_alignment = 1000
n_iterations = 50 

# Model configurations for 20 clients (ub, ua, dropout_rate)
models_params = [
    (10, 100, 0.1),
    (50, 100, 0.25),
    (70, 300, 0.15),
    (200, 70, 0.2),
    (128, 256, 0.1), 
    (256, 512, 0.15), 
    (93, 200, 0.25),
    (200, 400, 0.1),
    (240, 300, 0.15),
    (290, 340, 0.25),
    (10, 100, 0.1),
    (50, 100, 0.25),
    (70, 300, 0.15),
    (200, 70, 0.2),
    (128, 256, 0.1), 
    (256, 512, 0.15), 
    (93, 200, 0.25),
    (200, 400, 0.1),
    (240, 300, 0.15),
    (290, 340, 0.25)
]

print(f"Configuration: {n_parties} clients, input_shape={input_shape}, {n_iterations} iterations")

Configuration: 20 clients, input_shape=(100, 52), 50 iterations


In [8]:
# ============ LOAD PRIVATE DATASET (HomeOccupancy) ============
x_train, y_train, x_test, y_test, occupancy_label_names = load_and_preprocess_homeoccupancy(window_size=window_size)

# ============ LOAD PUBLIC DATASET (HomeHAR) ============
pub_x, pub_y, har_label_names = load_and_preprocess_homehar(window_size=window_size)

# ============ NORMALIZE ============
print("\nNormalizing data...")
# Flatten for scaling, then reshape back
train_shape = x_train.shape
test_shape = x_test.shape
pub_shape = pub_x.shape

scaling_data = MinMaxScaler()
x_train_flat = x_train.reshape(-1, n_features)
x_test_flat = x_test.reshape(-1, n_features)
pub_x_flat = pub_x.reshape(-1, n_features)

x_train_flat = scaling_data.fit_transform(x_train_flat)
x_test_flat = scaling_data.transform(x_test_flat)
pub_x_flat = scaling_data.transform(pub_x_flat)

x_train = x_train_flat.reshape(train_shape)
x_test = x_test_flat.reshape(test_shape)
pub_x = pub_x_flat.reshape(pub_shape)

# ============ SETUP LABELS ============
n_classes = 3  # HomeOccupancy has 3 classes: empty, sleep, work
original_labels = occupancy_label_names

# ============ PARTITION PRIVATE DATA OVER 10 CLIENTS ============
print("\nPartitioning private data over clients...")
pri_x_list, pri_y_list, pri_x_total, pri_y_total = split_dataset(
    x_train, y_train, 
    samples_per_class=n_samples_per_class,
    n_models=n_parties, 
    include_classes='all', 
    to_categorical_flag=True,
    n_classes_total=n_classes
)

y_train_cat = to_categorical(y_train, num_classes=n_classes)
y_test_cat = to_categorical(y_test, num_classes=n_classes)

# Shuffle data
shuffle_idx = np.random.permutation(len(x_train))
x_train = x_train[shuffle_idx]
y_train = y_train[shuffle_idx]
y_train_cat = y_train_cat[shuffle_idx]

shuffle_idx_pub = np.random.permutation(len(pub_x))
pub_x = pub_x[shuffle_idx_pub]
pub_y = pub_y[shuffle_idx_pub]

print(f'\nShape of x train data is: {x_train.shape}. Shape of y train data is: {y_train.shape}')
print(f'Shape of x test data is: {x_test.shape}. Shape of y test data is: {y_test.shape}')
print(f"Shape of x public data is: {pub_x.shape}. Shape of y public data is: {pub_y.shape}")
print(f"Single local x train is: {pri_x_list[0].shape}. Shape of single y train is: {pri_y_list[0].shape}")
print(f"\nPrivate dataset classes: {occupancy_label_names}")
print(f"Public dataset classes: {har_label_names}")

Loading HomeOccupancy dataset from HuggingFace...
  empty_1.csv: 2890 windows
  empty_2.csv: 2752 windows
  empty_3.csv (test): 2724 windows
  sleep_1.csv: 2868 windows
  sleep_2.csv: 2836 windows
  sleep_3.csv (test): 2720 windows
  work_1.csv: 1784 windows
  work_2.csv: 2633 windows
  work_3.csv (test): 2900 windows

HomeOccupancy loaded: 15763 train windows, 8344 test windows
Loading HomeHAR dataset from HuggingFace...
  data1/drink.csv: 3150 windows
  data1/eat.csv: 2974 windows
  data1/empty.csv: 2857 windows
  data1/sleep.csv: 2999 windows
  data1/smoke.csv: 2975 windows
  data1/watch.csv: 2851 windows
  data1/work.csv: 2863 windows

HomeHAR loaded: 20669 public windows

Normalizing data...

Partitioning private data over clients...

Shape of x train data is: (15763, 100, 52). Shape of y train data is: (15763,)
Shape of x test data is: (8344, 100, 52). Shape of y test data is: (8344,)
Shape of x public data is: (20669, 100, 52). Shape of y public data is: (20669,)
Single local x 

# Federated learning

## i.i.d. clients

In [14]:
iid_models = [get_model(n_classes, input_shape, model_params) for model_params in models_params] 

# print data shapes
print('train_batches shape : ', x_train.shape)
print('train_labels shape : ', y_train.shape)
print('pub_data_batches shape : ', pub_x.shape)
print('pub_data_labels shape : ', pub_y.shape)
print('test_batches shape : ', x_test.shape)
print('cat_test_labels shape : ', y_test_cat.shape)

print('pri_x_list shape : ', pri_x_list[0].shape)
print('pri_y_list shape : ', pri_y_list[0].shape)
print('pri_x_total shape : ', pri_x_total.shape)
print('pri_y_total shape : ', pri_y_total.shape)

train_batches shape :  (15763, 100, 52)
train_labels shape :  (15763,)
pub_data_batches shape :  (20669, 100, 52)
pub_data_labels shape :  (20669,)
test_batches shape :  (8344, 100, 52)
cat_test_labels shape :  (8344, 3)
pri_x_list shape :  (600, 100, 52)
pri_y_list shape :  (600, 3)
pri_x_total shape :  (12000, 100, 52)
pri_y_total shape :  (12000, 3)


In [15]:
print("Local training")
# Train on own private dataset only
local_models = [get_model(n_classes, input_shape, model_params )[0] for model_params in models_params] 
for i, model in enumerate(local_models) : 
    local_training_history = model.fit(pri_x_list[i], pri_y_list[i], validation_data = (x_test, y_test_cat), epochs = 20, verbose = True)
    pd.DataFrame(local_training_history.history).to_csv(os.path.join(experiment_dir, 'local_train_iid', 'local_training_{}.csv'.format(i)))
    break 
print() 
print("central training")
# Training a model on all distributed dataset (centralized training). 
centralized_models = [get_model(n_classes, input_shape, model_params )[0] for model_params in models_params] 
for i, model in enumerate(centralized_models) : 
    centralized_training_history = model.fit(pri_x_total, pri_y_total, validation_data = (x_test, y_test_cat), epochs = 20, verbose = True)
    pd.DataFrame(centralized_training_history.history).to_csv(os.path.join(experiment_dir, 'central_train_iid', 'centralized_training_{}.csv'.format(i)))
    break 


Local training
Epoch 1/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.3300 - loss: 1.1034 - val_accuracy: 0.3266 - val_loss: 1.1045
Epoch 2/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3350 - loss: 1.1042 - val_accuracy: 0.3266 - val_loss: 1.1044
Epoch 3/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3367 - loss: 1.1035 - val_accuracy: 0.3266 - val_loss: 1.1044
Epoch 4/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3367 - loss: 1.1039 - val_accuracy: 0.3266 - val_loss: 1.1044
Epoch 5/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3367 - loss: 1.1038 - val_accuracy: 0.3266 - val_loss: 1.1044
Epoch 6/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3367 - loss: 1.1039 - val_accuracy: 0.3266 - val_loss: 1.1044
Epoch 7/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3350 - loss: 1.1038 - val_accuracy: 0.3266 - val_loss: 1.1044
Epoch 8/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3350 - loss: 1.1037 - val_accur

In [16]:
# Use public data for distillation (logits from models, not categorical labels)
# The public dataset has 7 classes but we use the model's output logits for distillation
shared_public_dataset = (pub_x[:n_alignment, ...], pub_y[:n_alignment, ...])
validation_dataset = (x_test, y_test_cat)
fedmd_nodes = [FedAMDNode(iid_models[i], (pri_x_list[i], pri_y_list[i]), shared_public_dataset, target_validation_gen = validation_dataset) for i in range(n_parties)]

# Training iterations 
for iteration in range(n_iterations) : 
    print('\n Iteration:', iteration)

    for i, node in enumerate(fedmd_nodes) : 
        logger_file = os.path.join(experiment_dir,'iid', 'train_{}.csv'.format(i))
        node.train_on_target(epochs = 3, verbose = False, logger_file = logger_file, evaluate = True)

    # seed and alpha variables are used in FedAMDNode for mixup
    seed = np.random.randint(0, 10000) 
    alpha = np.random.rand()

    pub_scores, priv_performances = collect_metadatas(fedmd_nodes, seed, alpha) 
    print("models' accuracies:", priv_performances) 
    print("Avg Acc:{} Mean:{} Std:{}".format(np.mean(priv_performances), np.mean(pub_scores), np.std(pub_scores)))
  
    # Aggregate training metadata 
    weighted_pub_scores = aggregate_training_metadatas(pub_scores, priv_performances, weighted_averaging = True) 

    # Receive training metadata (and rebuilds Carrier dataset with updated labels)
    for i, node in enumerate(fedmd_nodes) : 
        node.receive_training_metadata(weighted_pub_scores)   
        node.train_on_public(epochs = 1, verbose = False)


 Iteration: 0
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
models' accuracies: [0.7708532810211182, 0.19103547930717468, 0.4688398838043213, 0.3048897385597229, 0.6342281699180603, 0.7885906100273132, 0.3583413362503052, 0.34755513072013855, 0.3394055664539337, 0.

## Non-i.i.d. clients

In [17]:
## Non-i.i.d. clients

# Non-IID class distribution for 20 clients with 3 classes (empty=0, sleep=1, work=2)

parties_classes = [

    [0, 1, 2],  # Client 0: all classes

    [0, 1],     # Client 1: empty, sleep

    [1, 2],     # Client 2: sleep, work

    [0, 2],     # Client 3: empty, work

    [0, 1, 2],  # Client 4: all classes

    [0],        # Client 5: empty only

    [1],        # Client 6: sleep only

    [2],        # Client 7: work only

    [0, 1],     # Client 8: empty, sleep

    [1, 2],     # Client 9: sleep, work

    [0, 1, 2],  # Client 10: all classes

    [0, 1],     # Client 11: empty, sleep

    [1, 2],     # Client 12: sleep, work

    [0, 2],     # Client 13: empty, work

    [0, 1, 2],  # Client 14: all classes

    [0],        # Client 15: empty only

    [1],        # Client 16: sleep only

    [2],        # Client 17: work only

    [0, 1],     # Client 18: empty, sleep

    [1, 2],     # Client 19: sleep, work

]

noniid_models = [get_model(n_classes, input_shape, model_params) for model_params in models_params] 

pri_x_list_noniid, pri_y_list_noniid, pri_x_total_noniid, pri_y_total_noniid = split_dataset(

    x_train, y_train, 

    samples_per_class=n_samples_per_class,

    n_models=n_parties, 

    include_classes=parties_classes, 

    to_categorical_flag=True,

    n_classes_total=n_classes

)

# print data shapes

print('train_batches shape : ', x_train.shape)

print('train_labels shape : ', y_train.shape)

print('pub_data_batches shape : ', pub_x.shape)

print('pub_data_labels shape : ', pub_y.shape)

print('test_batches shape : ', x_test.shape)

print('categorical test_labels shape : ', y_test_cat.shape)

print("_____________________________________________")

for i in range(n_parties):

    print(f'Client {i} (classes {parties_classes[i]}): pri_x shape: {pri_x_list_noniid[i].shape}, pri_y shape: {pri_y_list_noniid[i].shape}')

print("_____________________________________________")

print('pri_x_total shape : ', pri_x_total_noniid.shape)

print('pri_y_total shape : ', pri_y_total_noniid.shape)

train_batches shape :  (15763, 100, 52)
train_labels shape :  (15763,)
pub_data_batches shape :  (20669, 100, 52)
pub_data_labels shape :  (20669,)
test_batches shape :  (8344, 100, 52)
categorical test_labels shape :  (8344, 3)
_____________________________________________
Client 0 (classes [0, 1, 2]): pri_x shape: (600, 100, 52), pri_y shape: (600, 3)
Client 1 (classes [0, 1]): pri_x shape: (400, 100, 52), pri_y shape: (400, 3)
Client 2 (classes [1, 2]): pri_x shape: (400, 100, 52), pri_y shape: (400, 3)
Client 3 (classes [0, 2]): pri_x shape: (400, 100, 52), pri_y shape: (400, 3)
Client 4 (classes [0, 1, 2]): pri_x shape: (600, 100, 52), pri_y shape: (600, 3)
Client 5 (classes [0]): pri_x shape: (200, 100, 52), pri_y shape: (200, 3)
Client 6 (classes [1]): pri_x shape: (200, 100, 52), pri_y shape: (200, 3)
Client 7 (classes [2]): pri_x shape: (200, 100, 52), pri_y shape: (200, 3)
Client 8 (classes [0, 1]): pri_x shape: (400, 100, 52), pri_y shape: (400, 3)
Client 9 (classes [1, 2]):

In [18]:
print("Local training")
# Train on own private dataset only
local_models = [get_model(n_classes, input_shape, model_params )[0] for model_params in models_params] 
for i, model in enumerate(local_models) : 
    local_training_history = model.fit(pri_x_list_noniid[i], pri_y_list_noniid[i], validation_data = (x_test, y_test_cat), epochs = 20, verbose = True)
    pd.DataFrame(local_training_history.history).to_csv(os.path.join(experiment_dir, 'local_train_noniid', 'local_training_{}.csv'.format(i)))
    break 

print() 
print("central training")

# Training a model on all distributed dataset (centralized training). 
centralized_models = [get_model(n_classes, input_shape, model_params )[0] for model_params in models_params] 
for i, model in enumerate(centralized_models) : 
    centralized_training_history = model.fit(pri_x_total_noniid, pri_y_total_noniid, validation_data = (x_test, y_test_cat), epochs = 20, verbose = True)
    pd.DataFrame(centralized_training_history.history).to_csv(os.path.join(experiment_dir, 'central_train_noniid', 'centralized_training_{}.csv'.format(i)))
    break 


Local training
Epoch 1/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.3250 - loss: 1.0969 - val_accuracy: 0.3473 - val_loss: 1.0968
Epoch 2/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3317 - loss: 1.0959 - val_accuracy: 0.3476 - val_loss: 1.0960
Epoch 3/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3333 - loss: 1.0949 - val_accuracy: 0.3476 - val_loss: 1.0953
Epoch 4/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3350 - loss: 1.0941 - val_accuracy: 0.3476 - val_loss: 1.0945
Epoch 5/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3383 - loss: 1.0929 - val_accuracy: 0.3476 - val_loss: 1.0936
Epoch 6/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3383 - loss: 1.0920 - val_accuracy: 0.3476 - val_loss: 1.0927
Epoch 7/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3450 - loss: 1.0906 - val_accuracy: 0.3476 - val_loss: 1.0918
Epoch 8/20
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3433 - loss: 1.0898 - val_accur

In [19]:
shared_public_dataset = (pub_x[:n_alignment, ...], pub_y[:n_alignment, ...])
validation_dataset = (x_test, y_test_cat)
fedmd_nodes = [FedAMDNode(noniid_models[i], (pri_x_list_noniid[i], pri_y_list_noniid[i]), shared_public_dataset, target_validation_gen = validation_dataset) for i in range(n_parties)]

# Training iterations 
for iteration in range(n_iterations) : 
    print('\n Iteration:', iteration)

    for i, node in enumerate(fedmd_nodes) : 
        logger_file = os.path.join(experiment_dir,'noniid', 'train_{}.csv'.format(i))
        node.train_on_target(epochs = 3, verbose = False, logger_file = logger_file, evaluate = True)

    # seed and alpha variables are used in FedAMDNode for mixup
    seed = np.random.randint(0, 10000) 
    alpha = np.random.rand()

    pub_scores, priv_performances = collect_metadatas(fedmd_nodes, seed, alpha) 
    print("models' accuracies:", priv_performances) 
    print("Avg Acc:{} Mean:{} Std:{}".format(np.mean(priv_performances), np.mean(pub_scores), np.std(pub_scores)))
  
    # Aggregate training metadata 
    weighted_pub_scores = aggregate_training_metadatas(pub_scores, priv_performances, weighted_averaging = True) 

    # Receive training metadata (and rebuilds Carrier dataset with updated labels)
    for i, node in enumerate(fedmd_nodes) : 
        node.receive_training_metadata(weighted_pub_scores)   
        node.train_on_public(epochs = 1, verbose = False)


 Iteration: 0
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
models' accuracies: [0.3419223427772522, 0.6409395933151245, 0.34755513072013855, 0.38183125853538513, 0.7924256920814514, 0.32969799637794495, 0.32646211981773376, 0.34755513072013855, 0.3259827494621277,

# Plots results 

In [20]:


def smooth(signal, window_len = 11, polyorder = 3) : 
    return savgol_filter(signal, window_length= window_len, polyorder = polyorder)

def get_csv_files(dir) : 
    return [join(dir, f) for f in os.listdir(dir) if f.endswith('.csv')]


class Experiment : 

    def __init__(self, root) : 
        self.root = root 
        self.colors = list(mcolors.TABLEAU_COLORS.keys())
    
    
    def get_last_accuracies (self, subdir) : 
        return [pd.read_csv(f)['val_accuracy'].values[-1] for f in get_csv_files(join(self.root, subdir))]

    def get_accuracies(self, subdir) : 
        return [pd.read_csv(f)['val_accuracy'] for f in get_csv_files(join(self.root, subdir))]
    
    def plot_fedMD_like(self, left, center, right, labels, shades = None, title = 'Accuracy', limit = None):
        assert len(left) == len(right) == len(center), 'statistics should have the same length'
        
        n_epochs = len(center[0]) 
        epochs = np.arange(n_epochs) 
        n_parties = len(center)
        if limit is None : 
            limit = n_parties 
        plt.figure(figsize=(20, 11))

        plt.subplot(2, 2, 1)
        for i in range(limit) : 
            if left is not None : 
                plt.hlines(y=left[i], xmin=-10, xmax=10, linestyle = '--', color = self.colors[i])
            if right is not None : 
                plt.hlines(y=right[i], xmin=n_epochs-10, xmax=n_epochs+10, linestyle = '--', color = self.colors[i])
            plt.plot(epochs, center[i], label=labels[i], color = self.colors[i])
            if shades is not None :
                plt.fill_between(epochs, shades[0][i], shades[1][i], alpha=0.1, color = self.colors[i])
        plt.legend(loc='best', bbox_to_anchor=(0.95, 0.5))
        plt.title(title) 
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy')
        plt.xlim(0, n_epochs)
        plt.show()



def plot_fedMD_like_comparison(left, center, right, labels = None, shades = None, colors = None, title = None, limit = None) :
    n_epochs = len(center[0]) 
    epochs = np.arange(n_epochs) 
    n_parties = len(center)
    if limit is None : 
        limit = n_parties 
    plt.figure(figsize=(20, 11))
    if colors is None : 
        colors = list(mcolors.TABLEAU_COLORS.keys())
    plt.subplot(2, 2, 1)
    for i in range(limit) : 
        if left is not None :
            plt.hlines(y=left[i], xmin=-10, xmax=10, linestyle = '--', color = colors[i])
        if right is not None : 
            plt.hlines(y=right[i], xmin=n_epochs-10, xmax=n_epochs+10, linestyle = '--', color = colors[i])
        plt.plot(epochs, center[i], label=labels[i], color = colors[i])
        if shades is not None :
            plt.fill_between(epochs, shades[i][0], shades[i][0], alpha=0.1, color = colors[i])
    plt.legend(loc='lower right', bbox_to_anchor=(0.95, 0.5))
    plt.title(title) 
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.xlim(0, n_epochs)
    plt.show()


In [21]:

exp = Experiment(experiment_dir)
last_local_accs_iid = [0 for i in range(n_parties)] # exp.get_last_accuracies('local_train_iid')
last_local_accs_noniid = [0 for i in range(n_parties)] # exp.get_last_accuracies('local_train_noniid')

last_central_accs_iid = [0 for i in range(n_parties)] #exp.get_last_accuracies('central_train_iid')
last_central_accs_noniid = [0 for i in range(n_parties)] # exp.get_last_accuracies('central_train_noniid')

fedAMD_iid = exp.get_accuracies('iid') 
fedAMD_noniid = exp.get_accuracies('noniid') 
smooth_iid = [smooth(acc) for acc in fedAMD_iid]
smooth_noniid = [smooth(acc) for acc in fedAMD_noniid]

smooth_center_iid = [smooth(np.mean(fedAMD_iid, axis = 0), window_len = 27, polyorder = 3)]
smooth_center_noniid = [smooth(np.mean(fedAMD_noniid, axis = 0), window_len = 27, polyorder = 3)]

center_iid = [np.mean(fedAMD_iid, axis = 0)]
center_noniid = [np.mean(fedAMD_noniid, axis = 0)]

models_gains_iid = [int(round(fedAMD_iid[i].values[-1] - last_local_accs_iid[i], 2) *100) for i in range(len(last_local_accs_iid))]
models_gains_noniid = [int(round(fedAMD_noniid[i].values[-1] - last_local_accs_noniid[i], 2) *100) for i in range(len(last_local_accs_noniid))]
print('models gains iid:', models_gains_iid)
print('models gains noniid:', models_gains_noniid)

ts = [last_local_accs_iid, smooth_iid, last_central_accs_iid]
for t in ts : 
    print(len(t)) 
exp.plot_fedMD_like(last_local_accs_iid, smooth_iid, last_central_accs_iid, \
    labels = ['Model ' + str(i) for i in range(n_parties)],
     shades =None, 
     title = 'FedAKD accuracy on HomeOccupancy dataset (i.i.d)' , limit = 5)

exp.plot_fedMD_like(last_local_accs_noniid, smooth_noniid, last_central_accs_noniid, \
    labels = ['Model ' + str(i)  for i in range(n_parties)],
     shades =None, 
     title = 'FedAKD accuracy HomeOccupancy dataset (Non-i.i.d)', limit = 5)

NameError: name 'mcolors' is not defined